In [ ]:
import ee
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta
import time
import itertools
import os

In [ ]:
raw_path = os.path.join('..', 'data', 'raw')
caop_file = os.path.join(raw_path, 'Continente_CAOP2024_1.gpkg')

In [ ]:
# ==============================================================================
# 1. AUTENTICAÇÃO E INICIALIZAÇÃO (Não saltes isto)
# ==============================================================================
print("1. A iniciar o Google Earth Engine...")
try:
    ee.Initialize(project='gee-lisbon-housing') 
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='gee-lisbon-housing')

1. A iniciar o Google Earth Engine...


In [ ]:
# ==============================================================================
# 2. CARREGAMENTO E VALIDAÇÃO DE GEOMETRIAS (O teu lado local)
# ==============================================================================

# OPÇÃO A: Se tiver shapefile das freguesias
try:
    # Carregar shapefile (ajuste o caminho)
    gdf = gpd.read_file(caop_file)

    # filtrar apenas freguesias de lisboa
    gdf_lisboa = gdf[gdf['municipio'] == 'Lisboa'].copy()
    
    # Garantir que está em WGS84 (EPSG:4326)
    if gdf_lisboa.crs.to_epsg() != 4326:
        gdf_lisboa = gdf_lisboa.to_crs(epsg=4326)
    
    print(f"✅ {len(gdf_lisboa)} Freguesias de Lisboa carregadas e validadas.")

except Exception as e:
    print(f"❌ Erro ao processar shapefile: {e}")
    exit()

# Converter para FeatureCollection do Earth Engine
features = []
for _, row in gdf_lisboa.iterrows():
    features.append(
        ee.Feature(
            ee.Geometry(row.geometry.__geo_interface__),
            {'Freguesia': row['freguesia']} # Mantém o nome para a auditoria depois
        )
    )
freg_fc = ee.FeatureCollection(features)

c:\Users\migue\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'Continente_CAOP2024_1.gpkg': 'cont_areas_administrativas' (default), 'cont_distritos', 'cont_freguesias', 'cont_municipios', 'cont_nuts1', 'cont_nuts2', 'cont_nuts3', 'cont_trocos', 'layer_styles', 'inf_fonte_troco'. Specify layer parameter to avoid this warning.
  result = read_func(


✅ 24 Freguesias de Lisboa carregadas e validadas.


In [13]:
# ==============================================================================
# 3. CONFIGURAÇÃO DA COLEÇÃO VIIRS (O lado da Google)
# ==============================================================================

print("\n3. A preparar a coleção VIIRS...")
# Janeiro de 2015 a Dezembro de 2025 (Nota: 2025 só terá dados completos no futuro)
start_date = '2015-01-01'
end_date = '2026-01-01' # O end_date é exclusivo no GEE, por isso usamos 2026-01-01

viirs = (ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
         .filterDate(start_date, end_date)
         .select('avg_rad'))


3. A preparar a coleção VIIRS...


In [ ]:
# ==============================================================================
# 4. A FUNÇÃO DE REDUÇÃO (Mapeamento Temporal e Espacial)
# ==============================================================================
def extract_monthly_lights(image):
    # Extrai o Ano e Mês da imagem (YYYY-MM)
    date_str = ee.Date(image.get('system:time_start'))
    year = date_str.get('year')
    month = date_str.get('month')
    
    # Calcula a média por freguesia. 
    reduced = image.reduceRegions(
        collection=freg_fc,
        reducer=ee.Reducer.mean(),
        scale=463.83 
    )
    
    # Injeta a data em cada linha gerada
    return reduced.map(lambda f: f.set({
        'Year': year,
        'Month': month
    }))

# Aplica a função e achata a coleção para exportação tabular
viirs_final = viirs.map(extract_monthly_lights).flatten()

In [15]:
# ==============================================================================
# 5. EXPORTAÇÃO PARA O GOOGLE DRIVE COM MONITORIZAÇÃO
# ==============================================================================
print("\n4. A iniciar exportação para o Google Drive...")
task_name = 'VIIRS_Lisboa_Freguesias_2015_2025'

task = ee.batch.Export.table.toDrive(
    collection=viirs_final,
    description=task_name,
    folder='EarthEngine',
    fileNamePrefix=task_name,
    fileFormat='CSV',
    selectors=['Freguesia','Year', 'Month', 'mean'] # Força as colunas que importam
)

task.start()

# Loop para monitorizar o estado da tarefa em vez de ficares a adivinhar
while task.active():
    print(f"⏳ Tarefa a processar nos servidores da Google... Estado atual: {task.status()['state']}")
    time.sleep(30) # Espera 30 segundos antes de perguntar novamente

if task.status()['state'] == 'COMPLETED':
    print(f"✅ Exportação CONCLUÍDA! Vai ao teu Google Drive buscar o ficheiro '{task_name}.csv'")
else:
    print(f"❌ A exportação falhou: {task.status().get('error_message', 'Erro desconhecido')}")


4. A iniciar exportação para o Google Drive...
⏳ Tarefa a processar nos servidores da Google... Estado atual: READY
✅ Exportação CONCLUÍDA! Vai ao teu Google Drive buscar o ficheiro 'VIIRS_Lisboa_Freguesias_2015_2025.csv'


“Monthly mean nighttime radiance was computed for each parish using VIIRS-DNB data (500 m resolution), aggregated over administrative boundaries. This variable serves as a proxy for nocturnal economic and tourist activity intensity.”

In [ ]:
processed_path = os.path.join('..', 'data', 'processed', f'{task_name}.csv')

try:
    df_gee = pd.read_csv(processed_path)
    print(f"✅ Ficheiro lido com sucesso: {len(df_gee)} linhas encontradas.")
except FileNotFoundError:
    print("❌ Ficheiro CSV não encontrado. Descarregaste do Drive?")
    exit()

INÍCIO DA AUDITORIA DE INTEGRIDADE DOS DADOS
✅ Ficheiro lido com sucesso: 3168 linhas encontradas.


In [6]:
freguesias_lisboa = [
    'Ajuda', 'Alcântara', 'Alvalade', 'Areeiro', 'Arroios', 'Avenidas Novas', 
    'Beato', 'Belém', 'Benfica', 'Campo de Ourique', 'Campolide', 'Carnide', 
    'Estrela', 'Lumiar', 'Marvila', 'Misericórdia', 'Olivais', 'Parque das Nações', 
    'Penha de França', 'Santa Clara', 'Santa Maria Maior', 'Santo António', 
    'São Domingos de Benfica', 'São Vicente'
]

anos = list(range(2015, 2026)) # 2015 a 2025
meses = list(range(1, 13))

# Cruza Freguesias vs Anos vs Meses
combinacoes = list(itertools.product(freguesias_lisboa, anos, meses))
df_esperado = pd.DataFrame(combinacoes, columns=['Freguesia', 'Year', 'Month'])

# 3. Limpa e prepara os dados reais
df_gee_clean = df_gee.dropna(subset=['mean']).copy()

# 4. Comparações (Usa as 3 chaves agora)
df_faltas = df_esperado.merge(df_gee_clean, on=['Freguesia', 'Year', 'Month'], how='left', indicator=True)
dados_em_falta = df_faltas[df_faltas['_merge'] == 'left_only']

In [7]:
print("\n" + "=" * 80)
print("RESULTADOS DA AUDITORIA")
print("=" * 80)

if dados_em_falta.empty:
    print("🏆 EXCELENTE! Tens dados de radiância para todos os meses em todas as freguesias.")
else:
    print(f"🚨 ALARME: FALTAM {len(dados_em_falta)} REGISTOS!")
    print("Lista de falhas:")
    
    # Mostra os 20 primeiros erros para não inundar o terminal
    for _, row in dados_em_falta.head(20).iterrows():
        print(f" -> {row['Freguesia']} | Ano: {row['Year']} | Mês: {row['Month']}")
        
    if len(dados_em_falta) > 20:
        print(f"... e mais {len(dados_em_falta) - 20} falhas omitidas.")


RESULTADOS DA AUDITORIA
🏆 EXCELENTE! Tens dados de radiância para todos os meses em todas as freguesias.
